# Stage 1b — E6 cross-mechanism + E7 drive-swap  (CYCLED ×15, heat = TFF)

**Protocol (binding):** every stream = one corruption's severity-5 split
**cycled ×15** (~2355 steps). Single-pass 157-step streams are forbidden for
law measurement. **Hard** criterion primary; soft recorded separately.

**Reference (the only allowed):** `S_frozen` is read from
`analysis/stage1v2_S_frozen.json` (calibration cells, bracket-weighted, from
the verified restart campaign — expected ≈ 1.1063 ± 0.0095). Nothing from any
voided campaign seeds anything: existing `e6/e7_predictions.json` are INVALID
by default (`PREDICTIONS_POLICY="requarantine"` quarantines them and
re-freezes fresh predictions BEFORE any grid).

Provenance guards active: Drive-only RESULTS_DIR + sentinel, per-run protocol
fingerprints (checkpoint sha256, cycle_count, anchor_lambda / drive),
analyzer-VOID-without-reference, end-of-campaign file-count check.

**Budget:** cycled runs ≈ 380 s each on L4, ~7–9 runs/cell. E6 (4 cells) +
E7 (6 cells) ≈ one Stage-1-restart-class campaign (**~7–11 h L4 total**,
resumable, descending-η within each corruption). D5b (escape-point) is
zero-GPU and non-blocking.

**STOP after both verdicts; Stage 2 (controller, margin c=1.8 provisional)
awaits instruction.**

## 1. GPU check

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(out.stdout if out.returncode == 0 else "WARNING: no GPU.")

## 2. Config — `# === EDIT ME ===`

In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = "https://github.com/octadion/heat.git"
REPO_DIR   = "heat"
GIT_BRANCH = "formulation"

RESULTS_DIR = "/content/drive/MyDrive/pstar_results"   # must hold the cycled E1 campaign
CKPT_WRN    = "/content/drive/MyDrive/heat/experiments/checkpoints/wrn28_10_final.pt"
C10C_ROOT   = "data/cifar10c"
SEED, BATCH, WORKERS, CYCLES, BISECT = 42, 64, 2, 15, 3

RUN_E6, RUN_E7 = True, True

# RULE 3 — pre-registration validity. Existing e6/e7_predictions.json are
# INVALID by default (they may descend from the voided campaign):
#   "requarantine" (DEFAULT for this GO) -> quarantine + re-freeze fresh
#   "trust"        -> reuse (ONLY audit-confirmed / resuming THIS campaign)
#   "abort"        -> refuse and stop
PREDICTIONS_POLICY = "requarantine"
# NOTE: after the first session writes fresh predictions, RESUME with
# PREDICTIONS_POLICY = "trust" so the pre-registration is kept.
# =======================================================================
RULE_FLAGS = {"requarantine": ["--requarantine-predictions"],
              "trust": ["--trust-existing-predictions"],
              "abort": []}[PREDICTIONS_POLICY]
print("RESULTS_DIR =", RESULTS_DIR, "| policy =", PREDICTIONS_POLICY)

## 3. Mount Drive, clone, data, checkpoint

In [ ]:
import os, subprocess, hashlib
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(REPO_DIR):
    cmd = ["git", "clone"] + (["--branch", GIT_BRANCH] if GIT_BRANCH else []) + [REPO_URL, REPO_DIR]
    subprocess.run(cmd, check=True)
os.chdir("/content/" + REPO_DIR if not os.path.isabs(REPO_DIR) else REPO_DIR)
REPO_ROOT = os.getcwd()
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)
os.environ["PYTHONPATH"] = REPO_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
os.environ["PYTHONUTF8"] = "1"
subprocess.run(["python", "scripts/download_cifar10c.py", "--root", C10C_ROOT], check=True)
assert os.path.exists(CKPT_WRN), f"checkpoint missing: {CKPT_WRN}"
print("ckpt sha256[:16] =", hashlib.sha256(open(CKPT_WRN,'rb').read()).hexdigest()[:16])

## 4. Reference check — S_frozen must exist (the ONLY allowed reference)

In [ ]:
import json, os
sf = os.path.join(RESULTS_DIR, "analysis", "stage1v2_S_frozen.json")
assert os.path.exists(sf), ("stage1v2_S_frozen.json missing — run the restart "
                            "campaign's final diagnostic (S-FREEZE) first.")
d = json.load(open(sf))
print(f"S_frozen = {d.get('value', d.get('S_frozen'))} +/- {d.get('sigma')} "
      f"(n={d.get('n_points')})  <- expected ~1.1063 +/- 0.0095")

## 5. E6 — cross-mechanism (anchor), cycled ×15, resumable

Order enforced: reference-λ runs for all 4 cells → `e6_predictions.json`
((ηλ)_hat = S_frozen·η·‖ḡ‖) → at-λ̂ + λ grid + bisection (hard).
Re-run after a disconnect with `PREDICTIONS_POLICY="trust"`.

In [ ]:
import subprocess
if RUN_E6:
    rc = subprocess.run([
        "python", "scripts/run_stage1b_e6.py",
        "--results-dir", RESULTS_DIR, "--ckpt-wrn", CKPT_WRN,
        "--c10c-root", C10C_ROOT, "--cycles", str(CYCLES),
        "--bisect-steps", str(BISECT), "--seed", str(SEED),
        "--batch-size", str(BATCH), "--num-workers", str(WORKERS),
        *RULE_FLAGS]).returncode
    print("\n[E6] exit code", rc)
    if rc != 0:
        print("[E6] ABORTED by a guard — read the message; set "
              "PREDICTIONS_POLICY deliberately and re-run.")
else:
    print("[E6] skipped.")

### E6 verdict + plot

In [ ]:
import os
from IPython.display import Image, Markdown, display
adir = os.path.join(RESULTS_DIR, "analysis")
p = os.path.join(adir, "e6_crossmech.md")
if os.path.exists(p): display(Markdown(open(p, encoding="utf-8").read()))
p = os.path.join(adir, "e6_crossmech.png")
if os.path.exists(p): display(Image(filename=p))

## 6. E7 — drive-swap (entropy), cycled ×15, resumable

In [ ]:
import subprocess
if RUN_E7:
    rc = subprocess.run([
        "python", "scripts/run_stage1b_e7.py",
        "--results-dir", RESULTS_DIR, "--ckpt-wrn", CKPT_WRN,
        "--c10c-root", C10C_ROOT, "--cycles", str(CYCLES),
        "--bisect-steps", str(BISECT), "--seed", str(SEED),
        "--batch-size", str(BATCH), "--num-workers", str(WORKERS),
        *RULE_FLAGS]).returncode
    print("\n[E7] exit code", rc)
    if rc != 0:
        print("[E7] ABORTED by a guard — read the message above.")
else:
    print("[E7] skipped.")

### E7 verdict + plot

In [ ]:
import os
from IPython.display import Image, Markdown, display
adir = os.path.join(RESULTS_DIR, "analysis")
p = os.path.join(adir, "e7_driveswap.md")
if os.path.exists(p): display(Markdown(open(p, encoding="utf-8").read()))
p = os.path.join(adir, "e7_driveswap.png")
if os.path.exists(p): display(Image(filename=p))

## 7. D5b — escape-point analysis (parallel, ZERO GPU, never gates)

Runs any time (before/during/after E6/E7): drift level at escape (first step
with drift > 3× the cell's p_ref stationary plateau) for every collapsing
cycled-E1 run; per-corruption medians + contrast/others ratio.

In [ ]:
import subprocess
subprocess.run(["python", "scripts/analyze_d5b_escape.py",
                "--results-dir", RESULTS_DIR, "--cycles", str(CYCLES)])

## 8. STOP

Report both verdicts (`e6_crossmech.md`, `e7_driveswap.md`), the `[guard c]`
count-check lines, and `stage1v2_escape.md` (non-gating). **Stage 2
(controller, margin c=1.8 provisional, continuous validation) awaits
instruction.**